In [ ]:
import sys
from pathlib import Path

# Add the src directory to Python path
sys.path.insert(0, str(Path().resolve().parent / 'src'))

from config import EXTERNAL_DATA_DIR, INTERIM_DATA_DIR
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
from datetime import datetime

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

In [ ]:
musk_tweets = pd.read_parquet(EXTERNAL_DATA_DIR / "all_musk_posts.parquet")
print(f"Total tweets: {len(musk_tweets)}")
musk_tweets = musk_tweets[pd.to_datetime(musk_tweets['createdAt'], errors='coerce') >= '2023-01-01']
print(f"Tweets from 2023 onwards: {len(musk_tweets)}")

In [ ]:
# Check if there's a date column and convert it
date_cols = [col for col in musk_tweets.columns if 'date' in col.lower() or 'time' in col.lower()]
print(f"Date columns found: {date_cols}")

# Assuming there's a date column - adjust the column name as needed
if len(date_cols) > 0:
    musk_tweets['date'] = pd.to_datetime(musk_tweets[date_cols[0]], errors='coerce')
    musk_tweets['year_month'] = musk_tweets['date'].dt.to_period('M')
    
# Filter out null fullText
fullText = musk_tweets[musk_tweets['fullText'].notna()].copy()
print(f"Tweets with text: {len(fullText)}")

In [ ]:
# Load stop words from file
stop_words_file = Path().resolve().parent / 'data' / 'stop_words.txt'
with open(stop_words_file, 'r') as f:
    stop_words = set(word.strip().lower() for word in f.readlines() if word.strip())

print(f"Loaded {len(stop_words)} stop words")

def extract_keywords(text):
    if pd.isna(text):
        return []
    # Convert to lowercase and extract words
    words = re.findall(r'\b[a-z]{3,}\b', text.lower())
    # Filter out stop words
    return [word for word in words if word not in stop_words]

# Extract all keywords
all_keywords = []
for text in fullText['fullText']:
    all_keywords.extend(extract_keywords(text))

# Get top keywords
keyword_counts = Counter(all_keywords)
top_keywords = keyword_counts.most_common(20)

print("\nTop 20 Keywords:")
for word, count in top_keywords:
    print(f"{word}: {count}")

In [ ]:
# Plot top keywords
fig, ax = plt.subplots(figsize=(12, 6))
words, counts = zip(*top_keywords)
ax.barh(words, counts, color='steelblue')
ax.set_xlabel('Frequency', fontsize=12)
ax.set_title('Top 20 Keywords in Musk Tweets', fontsize=14, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Select interesting keywords to track over time
keywords_to_track = [word for word, _ in top_keywords[:10]]  # Top 10

# Create timeline data
fullText['keywords'] = fullText['fullText'].apply(extract_keywords)

# Convert createdAt to datetime and extract year-month
fullText['created_date'] = pd.to_datetime(fullText['createdAt'], errors='coerce')
fullText['year_month'] = fullText['created_date'].dt.to_period('M')

print(f"Date range: {fullText['created_date'].min()} to {fullText['created_date'].max()}")

# Count keywords per time period
timeline_data = []
for period in sorted(fullText['year_month'].dropna().unique()):
    period_tweets = fullText[fullText['year_month'] == period]
    period_keywords = []
    for keywords in period_tweets['keywords']:
        period_keywords.extend(keywords)
    
    keyword_counts_period = Counter(period_keywords)
    for keyword in keywords_to_track:
        timeline_data.append({
            'period': period,
            'keyword': keyword,
            'count': keyword_counts_period.get(keyword, 0)
        })

timeline_df = pd.DataFrame(timeline_data)
timeline_df['period'] = timeline_df['period'].astype(str)
print(f"\nTimeline data shape: {timeline_df.shape}")
timeline_df.head(10)

In [ ]:
# Plot keyword trends
fig, ax = plt.subplots(figsize=(16, 8))

for keyword in keywords_to_track:
    data = timeline_df[timeline_df['keyword'] == keyword].sort_values('period')
    ax.plot(data['period'], data['count'], marker='o', label=keyword, linewidth=2)

ax.set_xlabel('Time Period', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Keyword Trends Over Time in Musk Tweets', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()